In [1]:
!pip install -q medmnist
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

np.random.seed(42)
tf.random.set_seed(42)


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Experiment 8: Feature Extraction from Time Series using CNN
**Aim:** To extract features from time series using a 1D CNN and classify using the extracted features.

In [2]:
n_per_class, length = 600, 128
x_axis = np.linspace(0, 1, length)
data, labels = [], []
for _ in range(n_per_class):
    f = np.random.uniform(3, 6)
    ph = np.random.uniform(0, 2 * np.pi)
    data.append(np.sin(2 * np.pi * f * x_axis + ph)); labels.append(0)
    data.append(np.sign(np.sin(2 * np.pi * f * x_axis + ph))); labels.append(1)
    data.append(2 * ((f * x_axis + ph / (2 * np.pi)) % 1) - 1); labels.append(2)
data = np.array(data) + 0.2 * np.random.randn(len(data), length)
data = data[..., None]
labels = np.array(labels)

Xtr, Xte, ytr, yte = train_test_split(data, labels, test_size=0.2, random_state=42, stratify=labels)

inp = layers.Input(shape=(length, 1))
x = layers.Conv1D(16, 7, strides=1, padding='same', activation='relu')(inp)
x = layers.MaxPooling1D(pool_size=2, strides=2, padding='valid')(x)
x = layers.Conv1D(32, 5, strides=1, padding='same', activation='relu')(x)
x = layers.MaxPooling1D(pool_size=2, strides=2, padding='valid')(x)
x = layers.Conv1D(64, 3, strides=1, padding='same', activation='relu')(x)
feat = layers.GlobalAveragePooling1D(name='features')(x)
out = layers.Dense(3, activation='softmax')(feat)

cnn = models.Model(inp, out)
cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn.fit(Xtr, ytr, epochs=15, batch_size=32, validation_split=0.1, verbose=0)
print("CNN test accuracy:", cnn.evaluate(Xte, yte, verbose=0)[1])

extractor = models.Model(inp, cnn.get_layer('features').output)
Ftr = extractor.predict(Xtr, verbose=0)
Fte = extractor.predict(Xte, verbose=0)
print("Extracted feature shape:", Ftr.shape)

clf = LogisticRegression(max_iter=1000).fit(Ftr, ytr)
print("Logistic Regression accuracy on CNN features:", clf.score(Fte, yte))

CNN test accuracy: 1.0


Extracted feature shape: (1440, 64)
Logistic Regression accuracy on CNN features: 1.0
